# Week Problem Set: Data Wrangling

## Context
You are a Data Scientist at a regional health center. Patient data is divided across three systems:
- Administrative
- Lab Results
- Lifestyle Surveys

For this assignment, you will work with these datasets **independently** and focus only on **data wrangling tasks**.

---

## Datasets
1. patient_demographics.csv  
   - Patient ID, Age, Sex, Geography
     

2. clinical_data.csv  
   - Patient ID, Cholesterol, Blood Pressure ("120/80"), BMI
  

3. lifestyle_factors.csv  
   - Patient ID, Smoking, Diet, Heart Attack Risk (Target)

---

## Objectives
- Inspecting structure and quality
- Handling missing values
- Fixing data formats
- Identifying and treating anomalies

---

## Important Note
This is a guided notebook. You are expected to:
- Think critically
- Make decisions (and justify them)
- Explore different approaches

---

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

: 

Download the following CSV files from Google Classroom to your computer:

- patient_demographics.csv
- clinical_data.csv
- lifestyle_factors.csv

Run the code below and upload the files.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load datasets
demographics = pd.read_csv("/content/patient_demographics.csv")
clinical = pd.read_csv("/content/clinical_data.csv")
lifestyle = pd.read_csv("/content/lifestyle_factors.csv")

# Basic Inspection

## Tasks:
- View first few rows
- Check datatypes
- Look at summary statistics

## Questions:
- Are there obvious data issues?
- Are datatypes appropriate?

## Hint:
- Use `.head()`, `.info()`, `.describe()` for all three dataframes to get a full picture of the data quality.
- Pandas documentation for reference: https://pandas.pydata.org/docs/user_guide/10min.html

In [ ]:
#your code here
print(demographics.head())
print(demographics.info())
print(demographics.describe())

In [ ]:
#your code here
print(clinical.head())
print(clinical.info())
print(clinical.describe())

In [ ]:
#your code here
print(lifestyle.head())
print(lifestyle.info())
print(lifestyle.describe())

# Data Cleaning

## 1. Blood Pressure Column

### Task:
- Inspect the Blood Pressure column

### Questions:
- Is it stored as a number or string?
- Can we compute averages directly?

### Hint:
- Format looks like "120/80"
- Consider splitting into two columns
- https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html

### Consequences:
- Keeping as string -> limits analysis
- Splitting -> enables calculations but increases feature count

In [ ]:
clinical['Blood Pressure'].head(10)

In [ ]:
#your code here
bp_split = clinical['Blood Pressure'].str.split('/', expand=True)

In [ ]:
#your code here
clinical['Systolic'] = pd.to_numeric(bp_split[0])
clinical['Diastolic'] = pd.to_numeric(bp_split[1])

In [ ]:
#your code here
clinical = clinical.drop(columns=['Blood Pressure'])

print(clinical[['Systolic', 'Diastolic']].head())

## 2. Missing Values

### Task:
Identify columns with null values

### Questions:
- Which columns have missing values?
- Why might values be missing?

### Hint:
- Lifestyle → user did not answer (non-critical)
- Clinical → test not performed (critical)
- https://wesmckinney.com/book/data-cleaning

### Why Missing Data Matters
Missing data is not just an inconvenience, it can affect the **validity of your analysis**.

There are different types of missingness:
- **MCAR (Missing Completely at Random)** → no pattern (least problematic)
- **MAR (Missing at Random)** → depends on other variables
- **MNAR (Missing Not at Random)** → depends on the missing value itself (most problematic)

Understanding *why* data is missing helps decide how to handle it.

### Consequences of Different Choices

#### 1. Dropping Missing Values (`dropna`)
- Pros: simple, clean dataset  
- Cons: loses data, may introduce bias  

---

#### 2. Filling Missing Values (`fillna`)
- Pros: keeps dataset size, usable for modeling  
- Cons: adds assumptions, can distort data  

---

#### 3. Column-Specific Strategy
- Clinical → careful imputation or flag missing  
- Lifestyle → consider "Unknown" category  

### Key Takeaway:
There is **no single correct method**.  
The best approach depends on:
- Why data is missing  
- How much is missing  
- The impact on downstream analysis  


In [ ]:
#your code here
print("Missing values in Demographics:\n", demographics.isnull().sum())
print("\nMissing values in Clinical:\n", clinical.isnull().sum())
print("\nMissing values in Lifestyle:\n", lifestyle.isnull().sum())

In [ ]:
#your code here
clinical['Cholesterol'] = clinical['Cholesterol'].fillna(clinical['Cholesterol'].median())
clinical['BMI'] = clinical['BMI'].fillna(clinical['BMI'].median())

In [ ]:
#your code here
lifestyle['Smoking'] = lifestyle['Smoking'].fillna('Unknown')
lifestyle['Diet'] = lifestyle['Diet'].fillna('Unknown')

# Exploratory Data Analysis (EDA)

## Univariate Analysis: Age Distribution

Univariate analysis examines **one variable at a time**.
The goal is to understand each column's distribution, central tendency, and spread before comparing variables.

We split variables into two types:
- **Numerical** (continuous): Age, Cholesterol, BMI, Blood Pressure
- **Categorical** (discrete): Sex, Smoking, Diet, Country

---

## 1. Numerical Variables : Distributions

### Task:
Plot histograms and KDE (Kernel Density Estimate) curves for: Age, Cholesterol, BMI

### Questions:
- Which variables are normally distributed?
- Which variables are skewed left or right?
- Are there any unexpected spikes or gaps?
- Based on the distribution shape, would any feature likely need a transformation before modeling?

### Hint:
- `sns.histplot(data=df, x='Age', kde=True)` draws a histogram with a density curve
- `plt.axvline(df['Age'].mean(), color='red', linestyle='--', label='Mean')` adds a mean line
- Skewness > 1 or < -1 is considered highly skewed

## Why This Matters

Many statistical models, such as **Linear Regression, Logistic Regression, and Linear Discriminant Analysis** assume that numerical features are approximately **normally distributed**. When this assumption is violated, it can lead to:

- Biased coefficients  
- Unreliable p-values  
- Poor model performance  

Even though tree-based models (like Decision Trees) do not require normality, highly skewed features can still negatively affect **distance-based algorithms** such as **KNN** or **SVM**.

This is why understanding the shape of your distributions early in **Exploratory Data Analysis (EDA)** is important. It helps determine whether transformations are needed before modeling.

---

## Common Transformations and When to Use Them

### 1. Log Transform — `np.log1p(x)`
Used when data is **right-skewed** (long tail on the right), such as income or cholesterol levels in some populations.

- Compresses large values  
- Pulls distribution closer to normal  
- Uses `log1p` (log(1 + x)) to safely handle zero values  

---

### 2. Square Root Transform — `np.sqrt(x)`
A milder transformation compared to log.

- Useful for **moderately right-skewed data**  
- Common for count data (e.g., number of medications)  
- Preserves more of the original structure  

---

### 3. Box-Cox Transform — `scipy.stats.boxcox(x)`
A powerful, data-driven transformation.

- Finds optimal power (λ) to best approximate normality  
- Requires all values to be **strictly positive**  
- Special cases:
  - λ = 0 → log transform  
  - λ = 0.5 → square root transform  

---

### 4. Yeo-Johnson Transform — `sklearn.preprocessing.PowerTransformer(method='yeo-johnson')`
A flexible alternative to Box-Cox.

- Works with **zero and negative values**  
- More suitable for real-world medical datasets  
- Automatically learns best transformation  

---

### 5. Standardization (Z-score) — `sklearn.preprocessing.StandardScaler`
Transforms features to have:

- Mean = 0  
- Standard deviation = 1  

Notes:

- Does **not change distribution shape**  
- Useful when features are on different scales (e.g., Age vs Cholesterol)  
- Does not fix skewness  

---

### 6. Min-Max Normalization — `sklearn.preprocessing.MinMaxScaler`
Rescales features to a fixed range:

- Values are scaled to **[0, 1]**  
- Does not fix skewness  

---

### Reference links:
- https://towardsdatascience.com/top-3-methods-for-handling-skewed-data-1334e0debf45/
- https://www.geeksforgeeks.org/python/data-normalization-with-pandas/


In [ ]:
#your code here
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(demographics['Age'], kde=True, ax=axes[0], color='skyblue').set_title('Age Distribution')
sns.histplot(clinical['Cholesterol'], kde=True, ax=axes[1], color='salmon').set_title('Cholesterol Distribution')
sns.histplot(clinical['BMI'], kde=True, ax=axes[2], color='green').set_title('BMI Distribution')

plt.tight_layout()
plt.show()

## 2. Categorical Variables : Bar Charts

### Task:
Plot bar charts showing the frequency distribution of: Smoking Status, Diet Quality, and Top 10 Countries

### Questions:
- What proportion of patients smoke?
- Which diet category is most common?
- Which countries are most represented?

### Hint:
- `sns.countplot(data=df, x='Smoking', order=df['Smoking'].value_counts().index)` orders bars by frequency
- For Country, filter to top 10: `df['Country'].value_counts().head(10)`
- Rotate x-axis labels if needed: `plt.xticks(rotation=45)`

### Why This Matters:
Imbalanced categories (e.g., 90% male patients) can introduce bias into analyses. Knowing category proportions helps interpret group-level results accurately.

In [ ]:
#your code here
plt.figure(figsize=(15, 5))

# Smoking Status
plt.subplot(1, 2, 1)
sns.countplot(data=lifestyle, x='Smoking', palette='viridis')
plt.title('Smoking Status Frequency')

# Diet Quality
plt.subplot(1, 2, 2)
sns.countplot(data=lifestyle, x='Diet', palette='magma')
plt.title('Diet Quality Frequency')

plt.show()

# Bivariate Analysis : Features vs Target

## Context
Bivariate analysis explores the relationship **between two variables**.
Here, we are specifically interested in how each feature relates to `Heart Attack Risk` (our target).

This is where we start forming hypotheses:
- Do smokers have higher risk?
- Does diet quality affect risk?
- Does cholesterol level differ between risk groups?

---

## Categorical Features vs Heart Attack Risk (Grouped Bar Charts)

### Task:
For the categorical features: Smoking, Diet, Sex, create a grouped bar chart showing the count of At-Risk vs Not-At-Risk patients within each category.

### Questions:
- Do smokers show noticeably higher risk counts?
- Does diet quality appear protective against heart attack risk?
- Is there a difference in risk between male and female patients?

### Hint:
- `sns.countplot(data=df, x='Smoking', hue='Heart Attack Risk')` creates grouped bars automatically
- `hue` splits each bar group by the target variable
- To see proportions (not raw counts) within groups, compute:
  `df.groupby('Smoking')['Heart Attack Risk'].mean()` → gives risk rate per group

### Why This Matters:
Raw counts can be misleading if groups have unequal sizes. Consider also plotting proportions (risk rate) to compare groups fairly.

In [ ]:
#your code here
# First, merge demographics and lifestyle to have Sex and Risk in one DataFrame
df_merged = pd.merge(demographics, lifestyle, on='Patient ID')

# Define categorical features
features = ['Smoking', 'Diet', 'Sex']

# Create grouped bar charts (Counts)
plt.figure(figsize=(18, 6))

for i, col in enumerate(features, 1):
    plt.subplot(1, 3, i)
    sns.countplot(data=df_merged, x=col, hue='Heart Attack Risk')
    plt.title(f'{col} vs Heart Attack Risk')
    plt.legend(title='Heart Attack Risk', labels=['No Risk (0)', 'At Risk (1)'])

plt.tight_layout()
plt.show()

# Calculate proportions (Risk Rate) for deeper insight
print("--- Risk Proportions (Mean Risk per Category) ---")
for col in features:
    print(f"\nRisk Rate by {col}:")
    print(df_merged.groupby(col)['Heart Attack Risk'].mean())

# Correlation Analysis

## Context
Correlation measures **how strongly two numerical variables move together**.

### Some types of Correlation Coefficient:
| Coefficient | Use Case | Range |
|---|---|---|
| **Pearson** | Linear relationship between continuous variables | -1 to +1 |
| **Spearman** | Monotonic (ranked) relationship; robust to outliers | -1 to +1 |

### Interpretation Guide:
- |r| > 0.7 → Strong correlation  
- |r| 0.4–0.7 → Moderate correlation  
- |r| < 0.4 → Weak correlation  
- Sign (+ or -) tells direction: positive = both increase together, negative = one increases as other decreases

---

## Pearson Correlation Heatmap

### Task:
Calculate and visualize the Pearson correlation matrix for all numerical columns including the target.

### Questions:
- Which feature has the highest correlation with Heart Attack Risk?
- Are any features highly correlated with each other (multicollinearity)?
- Does Systolic BP correlate strongly with Diastolic BP? Why might that be expected?

### Hint:
- `df[numeric_cols].corr(method='pearson')` computes the correlation matrix
- `sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')` visualizes it
- `mask = np.triu(np.ones_like(corr_matrix))` removes the duplicate upper triangle

### Warning:
Pearson assumes linearity and is sensitive to outliers. For skewed distributions, consider Spearman instead.

In [ ]:
#your code here
# Split Blood Pressure into numeric Systolic and Diastolic columns
bp_split = clinical['Blood Pressure'].str.split('/', expand=True)
clinical['Systolic'] = pd.to_numeric(bp_split[0])
clinical['Diastolic'] = pd.to_numeric(bp_split[1])

# Merge all three datasets on Patient ID
df_final = demographics.merge(clinical, on='Patient ID').merge(lifestyle, on='Patient ID')

# Define numerical columns (excluding ID)
numeric_cols = df_final.select_dtypes(include=[np.number]).columns.tolist()
if 'Patient ID' in numeric_cols:
    numeric_cols.remove('Patient ID')


# Calculate Pearson Correlation
pearson_corr = df_final[numeric_cols].corr(method='pearson')

# Plot Heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
sns.heatmap(pearson_corr, mask=mask, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Pearson Correlation Matrix')
plt.show()

# Spearman Correlation

### Task:
Repeat the correlation analysis using Spearman's rank correlation.

### Questions:
- Are the Spearman results significantly different from Pearson?
- If they differ, what might that suggest about the data's linearity or outliers?

### Hint:
- Use `method='spearman'` in `.corr()`
- Compare the two heatmaps side by side using `plt.subplots(1, 2)`

### Interpretation:
Large differences between Pearson and Spearman suggest the relationship is non-linear or that outliers are influencing Pearson.

In [ ]:
#your code here
# Calculate Spearman Correlation
spearman_corr = df_final[numeric_cols].corr(method='spearman')

# Plot Heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(spearman_corr, dtype=bool))
sns.heatmap(spearman_corr, mask=mask, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Spearman Correlation Matrix')
plt.show()

## Target Variable Analysis: Heart Attack Risk

## Context
Target variable is the variable that the user would want to predict using the rest of the dataset.

`Heart Attack Risk` is a **binary variable**: 0 (No Risk) and 1 (At Risk).

## Tasks:
- Count how many patients fall into each class
- Visualize the class distribution with a bar chart

## Hint:
- Use `.value_counts()` to get counts
- Use `sns.countplot()` or `plt.bar()` for visualization


In [ ]:
#your code here
print("Value Counts for Heart Attack Risk:")
print(lifestyle['Heart Attack Risk'].value_counts())

sns.countplot(data=lifestyle, x='Heart Attack Risk', palette='Set2')
plt.title('Distribution of Heart Attack Risk (Target)')
plt.show()

# Final Reflection

## Answer the following:

### 1. Missing Values
- How did you handle missing values in each dataset?
- Why did you choose that approach (drop, fill, or mixed strategy)?

---

### 2. Data Quality Issues
- Did you find any unrealistic or inconsistent values?
- How did you identify them?
- What action did you take (remove, clip, correct, or keep)?

---

### 3. Insights from EDA
- What patterns did you observe in the data?
- Did you find any relationships between variables (e.g., age, cholesterol, risk)?
- Were there any surprising findings or trends?

---

### 4. Correlation Findings
- Were Pearson and Spearman results similar or different? What does that suggest?
- Did you find any multicollinearity (two features highly correlated with each other)?

---

### 5. Improvements
- If you had more time, what would you improve in your data cleaning process?
- Would you try different strategies for missing values or outliers?
- What additional checks or visualizations would you add?

---

### Tip
There is no single correct answer here. Focus on:
- Justifying your decisions  
- Explaining trade-offs  
- Demonstrating understanding of data quality impact  

### Answer here:

1. Missing Values

    Action: Used a mixed strategy: Median for clinical numbers and "Unknown" for lifestyle text.  

    Reason: Medians are outlier-resistant, and "Unknown" avoids inventing patient habits while keeping all rows for analysis.  

2. Data Quality Issues

    Action: Identified that Blood Pressure was a string ("120/80") using .info().  

    Correction: Split it into numeric Systolic and Diastolic columns to allow for math and correlation.  

3. Insights from EDA

    Distributions: Age, Cholesterol, and BMI are mostly normal.  

    Surprises: Common risk factors like smoking and poor diet showed almost identical risk rates (~35-36%) to their healthy counterparts in this data.  

4. Correlation Findings

    Comparison: Pearson and Spearman were nearly identical, meaning no hidden non-linear patterns or major outlier interference.  

    Multicollinearity: None found; even the two BP metrics were surprisingly independent (r < 0.10).  

5. Improvements

    Strategy: Use KNN Imputation for more accurate missing value filling.  

    Visuals: Add boxplots to see if the "At Risk" group has a higher median cholesterol than the "No Risk" group, which histograms can miss.